# Day 5 - ReAct Agent Tool with Guardrails

> This is extention of Day 4 with Guardrails added to validate the tool calls

# Section-A - Setup (LLM+loading env)

In [1]:
from aiohttp.web_middlewares import middleware
from dotenv import load_dotenv
from langchain_classic.agents import AgentType
from langchain_classic.chains.question_answering.map_reduce_prompt import messages
from langchain_core.tools import tool

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Section B - Embeddings and Vector DB retrival

1. loading the file
2. chunking
3. embeddings
4. Vector DB ( Retrieval )


In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = r"C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf"

docs = PyPDFLoader(PDF_PATH).load()

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

In [5]:
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="nomic-embed-text")

In [6]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks,emb)
retriever = db.as_retriever(search_kwargs={"k":3})
print(len(chunks))

8


# Section 3 - Lets build the tools

1. calculator tool
2. data retrieval tool
3. Add Validators for tool calls
4. Use a Guardrail between llm and tool

In [7]:
from langchain.tools import tool

In [8]:
# Calculator tool

@tool
def calculator(expression:str) -> str:
    """Use this for math. Input should be a valid python math expression like '25*4+10'"""
    try:
        print(f"The calculator tool is called for {expression}")
        return str(eval(expression))
    except Exception as e:
        return f"Calculator Error: {e}"

In [9]:
# Document Search tool

@tool
def search_docs(query:str) -> str:
    """Use this to search the loaded documents and return relevant excerpts"""
    print(f"Search tool is called for {query}")
    results = retriever.invoke(input=query)
    blocks=[]
    for i,d in enumerate(results, 1):
        src = d.metadata.get("source","unknown")
        page = d.metadata.get("page","n/a")
        blocks.append(f"[{i}](source={src},page={page})\n{d.page_content})")
    return "\n\n".join(blocks)

# Lets add Tool Validators

In [10]:
import ast

def validate_tool_call(tool_name: str, tool_args: dict) -> (bool, str):
    """
    Check if a proposed tool call is valid.
    Return (True, "OK") if safe, else (False, "Error message").
    """
    print("Validating tool call")
    # Example: Guard for calculator tool
    if tool_name == "calculator":
        expr = tool_args.get("expression", "")
        try:
            ast.parse(expr, mode="eval")  # validate Python math syntax
        except SyntaxError as e:
            return False, f"⚠️ Invalid math expression: {expr}"
        return True, "OK"

    # Allow other tools by default
    return True, "OK"

# Lets add Middleware Guardrails for a tool call

In [11]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
@wrap_tool_call
def guard_tool_execution(request, handler):
    # request.tool_call is a dict like
    # {"name": "calculator", "args": {...}, ...}
    tool_call = request.tool_call
    tool_name = tool_call["name"]
    tool_args = tool_call.get("args", {})
    print("Executing the guardrail")
    # Validate the planned tool call
    allowed, message = validate_tool_call(tool_name, tool_args)
    if not allowed:
        # Return a ToolMessage instead of executing
        return ToolMessage(
            content=message,
            tool_call_id=tool_call.get("id", ""),
            status="error"
        )

    # If allowed → let the normal tool execution proceed
    return handler(request)

# Section 4 - Build React Agent

In [12]:
from langchain.agents import create_agent
tools = [calculator, search_docs]
agent = create_agent(
    tools=tools,
    model=llm,
    system_prompt="""
You are an intelligent assistant with access to tools.

1. If the user query is a **pure math expression** (contains numbers and math operators like +, -, *, /, **, parentheses):
   • Convert it into valid Python math syntax.
   • Call the calculator tool **once** with that expression.
   • Return the numeric result as a natural language answer.

2. For **non-math queries**:
   • First use the search_docs tool with the user’s question.
   • If the search_docs result contains relevant content, use it to answer the question.
   • If the search_docs result is not relevant or does not contain useful information, reply: “Document doesn’t have relevant info” then answer from your general knowledge.
   • Only call search_docs once per query.

Do not call any tool more than once. Always produce a final natural language answer.
""",
    middleware=[guard_tool_execution]
)

In [13]:
result = agent.invoke(
    {"messages":[{"role": "user", "content": "What is AccuMO"}]}
)
print(result["messages"][-1].content)

Executing the guardrail
Validating tool call
Search tool is called for AccuMO
AccuMO is a framework for optimizing the accuracy of multitask offloading in edge-assisted mobile augmented reality (AR) environments. It uses a two-level control loop to schedule multiple latency-critical tasks, such as depth estimation and odometry, to maximize overall accuracy. However, it has limitations, including its evaluation using a synthetic dataset and its assumption of a relatively stable network.


In [14]:
print(result["messages"][3].usage_metadata)

{'input_tokens': 909, 'output_tokens': 78, 'total_tokens': 987}


In [15]:
tests = [
    "What is 25 + (13 * 2)?",
    "Compute (3^5) * 10",
    "Explain semantic search and how it differs from keyword search, answer in 1 sentence",
    "According to the document, what is semantic search?, answer in 1 sentence",
    "Explain semantic search and how it differs from keyword search, answer in 1 sentence",
    "According to the document, what is semantic search? answer in 1 sentence"
]


In [16]:
for test in tests:
    result = agent.invoke(
        {"messages":[{"role": "user", "content": f"{test}"}]}
    )
    print(result["messages"][-1].content)
    print(result["messages"][-1].usage_metadata)
    print("\n")

Executing the guardrail
Validating tool call
The calculator tool is called for 25 + (13 * 2)
The result of the expression 25 + (13 * 2) is 51.
{'input_tokens': 520, 'output_tokens': 19, 'total_tokens': 539}


Executing the guardrail
Validating tool call
The calculator tool is called for (3**5) * 10
The result of the computation is 2430.
{'input_tokens': 519, 'output_tokens': 11, 'total_tokens': 530}


Executing the guardrail
Validating tool call
Search tool is called for Explain semantic search and how it differs from keyword search
Semantic search is a type of search technology that aims to return results that are most relevant to the user's intent, rather than just matching specific keywords, by analyzing the meaning and context of the search query.
{'input_tokens': 923, 'output_tokens': 42, 'total_tokens': 965}


Executing the guardrail
Validating tool call
Search tool is called for semantic search definition in document
Document doesn’t have relevant info. Semantic search is a type